# 02: Benchmarks

Two OLS benchmarks fit on metaorder training data before building the MLP.
These set the performance floor the neural network needs to exceed.

**Benchmark 1: Power law**
```
log I = alpha + delta * log Q_norm
```

**Benchmark 2: Almgren-Chriss**
```
log I = alpha + delta * log Q_norm + beta * log sigma_daily
```

Theory predicts delta = 0.5 and beta = 1.0.
Both models are fit on mo_train.parquet and evaluated on mo_test.parquet.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

DATA_DIR = Path("../data/processed")

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

## Load

In [ ]:
train = pd.read_parquet(DATA_DIR / "mo_train.parquet")
test  = pd.read_parquet(DATA_DIR / "mo_test.parquet")

print(f"Train: {len(train):,} metaorders")
print(f"Test : {len(test):,} metaorders")
print(f"Columns: {list(train.columns)}")

## Prepare

Work in log space. Keep only metaorders with positive I, Q_norm, and sigma_daily.
Remove outliers beyond 3 sigma in log I.

In [ ]:
def prepare(df: pd.DataFrame) -> pd.DataFrame:
    df = df[(df["I"] > 0) & (df["Q_norm"] > 0) & (df["sigma_daily"] > 0)].copy()
    df["log_I"]       = np.log(df["I"])
    df["log_Q"]       = np.log(df["Q_norm"])
    df["log_sigma"]   = np.log(df["sigma_daily"])
    df["log_V"]       = np.log(df["V_daily"])
    df["log_n_child"] = np.log(df["n_child"])
    mu, sd = df["log_I"].mean(), df["log_I"].std()
    df = df[np.abs(df["log_I"] - mu) < 3 * sd].reset_index(drop=True)
    return df

train_p = prepare(train)
test_p  = prepare(test)

print(f"Train after prep: {len(train_p):,}")
print(f"Test after prep : {len(test_p):,}")
print()
print("log_I distribution (train):")
print(train_p["log_I"].describe().round(3))
print()
print("log_sigma distribution (train):")
print(train_p["log_sigma"].describe().round(3))

## Benchmark 1: Power law

log I = alpha + delta * log Q_norm

In [ ]:
y_train = train_p["log_I"].values
y_test  = test_p["log_I"].values

X_train_b1 = train_p[["log_Q"]].values
X_test_b1  = test_p[["log_Q"]].values

b1 = LinearRegression().fit(X_train_b1, y_train)
delta_b1 = b1.coef_[0]

slope, intercept, r, p, se = stats.linregress(train_p["log_Q"].values, y_train)

print(f"  delta     : {delta_b1:.4f}")
print(f"  95% CI    : [{slope - 1.96*se:.4f}, {slope + 1.96*se:.4f}]")
print(f"  intercept : {b1.intercept_:.4f}")
print(f"  Train R2  : {r2_score(y_train, b1.predict(X_train_b1)):.4f}")

## Benchmark 2: Almgren-Chriss

log I = alpha + delta * log Q_norm + beta * log sigma_daily

In [ ]:
X_train_b2 = train_p[["log_Q", "log_sigma"]].values
X_test_b2  = test_p[["log_Q", "log_sigma"]].values

b2 = LinearRegression().fit(X_train_b2, y_train)
delta_b2 = b2.coef_[0]
beta_b2  = b2.coef_[1]

print(f"  delta     : {delta_b2:.4f}  (expected 0.5)")
print(f"  beta      : {beta_b2:.4f}  (expected 1.0)")
print(f"  intercept : {b2.intercept_:.4f}")
print(f"  Train R2  : {r2_score(y_train, b2.predict(X_train_b2)):.4f}")

## Constrained benchmark: delta = 0.5

Impose the theoretical exponent and fit only the intercept.

In [ ]:
y_train_resid         = y_train - 0.5 * train_p["log_Q"].values
intercept_constrained = y_train_resid.mean()
y_pred_constrained    = intercept_constrained + 0.5 * test_p["log_Q"].values

print(f"  intercept : {intercept_constrained:.4f}")

## Out-of-sample evaluation

In [ ]:
def evaluate(name: str, y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    mse  = mean_squared_error(y_true, y_pred)
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    bias = (y_pred - y_true).mean()
    print(f"  {name:<35} MSE={mse:.4f}  MAE={mae:.4f}  R2={r2:.4f}  Bias={bias:.4f}")
    return {"model": name, "mse": mse, "mae": mae, "r2": r2, "bias": bias}

print("Out-of-sample results:")
results = [
    evaluate("Power law (delta=0.5, constrained)", y_test, y_pred_constrained),
    evaluate("Power law (delta fitted)",           y_test, b1.predict(X_test_b1)),
    evaluate("Almgren-Chriss",                     y_test, b2.predict(X_test_b2)),
]

results_df = pd.DataFrame(results)
results_df.to_csv("../outputs/benchmark_results.csv", index=False)

## Residuals vs fitted

In [ ]:
y_pred_b2 = b2.predict(X_test_b2)
residuals = y_test - y_pred_b2

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(y_pred_b2, residuals, s=1, alpha=0.1, color="steelblue")
axes[0].axhline(0, color="firebrick", linewidth=1)
axes[0].set_xlabel("Predicted log I")
axes[0].set_ylabel("Residual")
axes[0].set_title("Almgren-Chriss residuals vs fitted")

axes[1].hist(residuals, bins=100, color="steelblue", edgecolor="none")
axes[1].set_xlabel("Residual")
axes[1].set_title("Residual distribution")

plt.tight_layout()
plt.savefig("../outputs/figures/benchmark_residuals.png", bbox_inches="tight")
plt.show()

print(f"Residual std  : {residuals.std():.4f}")
print(f"Residual skew : {pd.Series(residuals).skew():.4f}")

## Residuals vs Q_norm

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

ax.scatter(test_p["log_Q"].values, residuals, s=1, alpha=0.05, color="steelblue")
ax.axhline(0, color="firebrick", linewidth=1)

test_p_copy = test_p.copy()
test_p_copy["residual"] = residuals
test_p_copy["q_bin"]    = pd.qcut(test_p_copy["log_Q"], q=30, duplicates="drop")
bin_means   = test_p_copy.groupby("q_bin", observed=True)["residual"].mean()
bin_centers = test_p_copy.groupby("q_bin", observed=True)["log_Q"].median()
ax.plot(bin_centers.values, bin_means.values,
        color="firebrick", linewidth=2, label="Binned mean residual")

ax.set_xlabel("log(Q_norm)")
ax.set_ylabel("Residual")
ax.set_title("Almgren-Chriss residuals vs trade size")
ax.legend()

plt.tight_layout()
plt.savefig("../outputs/figures/residuals_vs_q.png", bbox_inches="tight")
plt.show()

## Summary

In [ ]:
print(f"Power law delta (fitted) : {delta_b1:.4f}")
print(f"Almgren-Chriss delta     : {delta_b2:.4f}  (expected 0.5)")
print(f"Almgren-Chriss beta      : {beta_b2:.4f}  (expected 1.0)")
print()
print("OOS MSE:")
for row in results:
    print(f"  {row['model']:<35} {row['mse']:.4f}")